In [10]:
import os

import load_dotenv
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

load_dotenv.load_dotenv()

True

In [24]:
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment_name = "MLflow Integration with Dagster"
try:
    mlflow.set_experiment(experiment_name)
except Exception as e:
    print(f"Error setting MLflow experiment: {e}")
    mlflow.create_experiment(experiment_name)
    mlflow.set_experiment(experiment_name)

MODEL_NAME_TO_REGISTER = "rental_forecast_model"

client = MlflowClient(MLFLOW_TRACKING_URI)

In [37]:
best_3_runs = client.search_runs(
    experiment_ids=[
        client.get_experiment_by_name(experiment_name).experiment_id
    ],
    max_results=3,
    filter_string="metrics.r2 > 0.8",
    order_by=["metrics.r2 DESC"],
)

In [38]:
for run in best_3_runs:
    print(run)
    break

<Run: data=<RunData: metrics={'mae': 60.41477120047367, 'r2': 0.8208542282093115, 'rmse': 93.32479125542153}, params={'colsample_bytree': '0.8',
 'learning_rate': '0.05',
 'max_depth': '6',
 'model_name': 'XGBRegressor',
 'n_estimators': '300',
 'random_state': '42',
 'subsample': '0.8'}, tags={'mlflow.runName': 'train_xgboost',
 'mlflow.source.name': '/Users/JayeshManani/Downloads/MLOps-Level3/.venv/lib/python3.12/site-packages/dagster/__main__.py',
 'mlflow.source.type': 'LOCAL',
 'mlflow.user': 'JayeshManani'}>, info=<RunInfo: artifact_uri='mlflow-artifacts:/1/1ef73a030244472db8eac7be6a644b26/artifacts', end_time=1780999104751, experiment_id='1', lifecycle_stage='active', run_id='1ef73a030244472db8eac7be6a644b26', run_name='train_xgboost', start_time=1780999104194, status='FINISHED', user_id='JayeshManani'>, inputs=<RunInputs: dataset_inputs=[], model_inputs=[]>, outputs=<RunOutputs: model_outputs=[<LoggedModelOutput: model_id='m-94cd08eb72704c2e8ad656957f62d455', step=0>]>>


In [39]:
best_run = best_3_runs[0]
best_run_id = best_run.info.run_id

In [40]:
best_run_id

'1ef73a030244472db8eac7be6a644b26'

In [41]:
best_run.data.params.get("model_name")

'XGBRegressor'

In [42]:
latest_version = client.get_latest_versions(MODEL_NAME_TO_REGISTER)

/var/folders/_b/v2z7twxj4bbczz_fz9lgtb2r0000gx/T/ipykernel_84649/2995014662.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_version = client.get_latest_versions(MODEL_NAME_TO_REGISTER)


In [43]:
for v in latest_version:
    print(f"Version: {v.version}, Stage: {v.current_stage}")

Version: 1, Stage: None


In [22]:
client.set_registered_model_alias(
    name=MODEL_NAME_TO_REGISTER, version=1, alias="champion"
)

In [23]:
client.delete_registered_model(name=MODEL_NAME_TO_REGISTER)